In [1]:
tabla='hmhab10'
dim='dim_hoshab'

In [2]:
import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [3]:


oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM hmhab10", con=connection2)

connection2.close()




In [4]:
base2

,oricenasicod,cenasicod,arehoscod,servhoscod,estenfcod,habcod,habubides,tiphabcod,esthabcod,habestregcod
0,1,076,03,D11,04,01,"A,B,C",3,1,0
1,1,076,03,D11,04,03,"A,B,C,D",1,1,0
2,1,076,03,D11,04,04,"A,B,C,D",1,1,0
3,1,076,03,D11,04,05,"A,B,C,D",1,1,0
4,1,076,03,C11,03,02,"A,B",1,1,1
...,...,...,...,...,...,...,...,...,...,...
16666,1,262,03,AB1,88,D3,1ER PISO,1,1,1
16667,1,262,03,AB1,88,D4,1ER PISO,1,1,1
16668,1,349,03,AB1,30,01,,1,1,0
16669,1,392,03,A24,03,H008,CUARTO PISO,1,1,1


In [5]:
base2.columns

Index(['oricenasicod', 'cenasicod', 'arehoscod', 'servhoscod', 'estenfcod',
       'habcod', 'habubides', 'tiphabcod', 'esthabcod', 'habestregcod'],
      dtype='object')

In [6]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


query_count_before = "SELECT COUNT(*) FROM hmhab10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla hmhab10 antes de la inserción: {cant_antes}")


#connection3.execute('CREATE TEMPORARY TABLE tmp_hmcam10 ()')
base2.to_sql(name='tmp_hmhab10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL hmcam10 FALSO CON LO OBTENIDO DEL ORACLE

query="""
ALTER TABLE tmp_hmhab10
ALTER COLUMN oricenasicod  TYPE character(1),
ALTER COLUMN cenasicod  TYPE character(3),
ALTER COLUMN arehoscod  TYPE character(2),
ALTER COLUMN servhoscod  TYPE character(3),
ALTER COLUMN estenfcod  TYPE character(2),
ALTER COLUMN habcod  TYPE character(4),
ALTER COLUMN habubides  TYPE character(20),
ALTER COLUMN tiphabcod  TYPE character(1),
ALTER COLUMN esthabcod  TYPE character(1),
ALTER COLUMN habestregcod  TYPE character(1);



UPDATE hmhab10
SET
arehoscod = t.arehoscod,
servhoscod = t.servhoscod,
habubides = t.habubides,
tiphabcod = t.tiphabcod,
esthabcod = t.esthabcod,
habestregcod = t.habestregcod

FROM tmp_hmhab10 t 
WHERE hmhab10.oricenasicod = t.oricenasicod AND TRIM(hmhab10.oricenasicod) <> '' AND hmhab10.oricenasicod IS NOT NULL AND
hmhab10.cenasicod = t.cenasicod AND TRIM(hmhab10.cenasicod) <> '' AND hmhab10.cenasicod IS NOT NULL AND
hmhab10.arehoscod = t.arehoscod AND TRIM(hmhab10.arehoscod) <> '' AND hmhab10.arehoscod IS NOT NULL AND
hmhab10.servhoscod = t.servhoscod AND TRIM(hmhab10.servhoscod) <> '' AND hmhab10.servhoscod IS NOT NULL AND
hmhab10.estenfcod = t.estenfcod AND TRIM(hmhab10.estenfcod) <> '' AND hmhab10.estenfcod IS NOT NULL AND
hmhab10.habcod = t.habcod AND TRIM(hmhab10.habcod) <> '' AND hmhab10.habcod IS NOT NULL;


INSERT INTO hmhab10 
(oricenasicod, cenasicod, arehoscod, servhoscod, estenfcod, habcod, habubides, tiphabcod, esthabcod, habestregcod) 
SELECT 
oricenasicod, cenasicod, arehoscod, servhoscod, estenfcod, habcod, habubides, tiphabcod, esthabcod, habestregcod

FROM tmp_hmhab10  t
WHERE NOT EXISTS (
    SELECT 1 
    FROM hmhab10 
    WHERE hmhab10.oricenasicod = t.oricenasicod and hmhab10.oricenasicod IS NOT NULL AND
    hmhab10.cenasicod = t.cenasicod and hmhab10.cenasicod IS NOT NULL AND
    hmhab10.arehoscod = t.arehoscod and hmhab10.arehoscod IS NOT NULL AND
    hmhab10.servhoscod = t.servhoscod and hmhab10.servhoscod IS NOT NULL AND
    hmhab10.estenfcod = t.estenfcod and hmhab10.estenfcod IS NOT NULL AND
    hmhab10.habcod = t.habcod and hmhab10.habcod IS NOT NULL
) ;


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_hmhab10;
SELECT COUNT(*) FROM hmhab10;
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla hmhab10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")
base2 = pd.read_sql_query(f"SELECT * FROM hmhab10", con=connection3)


connection3.close()


Cantidad de filas en la tabla hmhab10 antes de la inserción: 16432
Cantidad de filas en la tabla hmhab10 después de la inserción: 16671
La cantidad de filas insertadas fue de: 239


In [7]:
#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)

#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS
query_count_before = f"SELECT COUNT(*) FROM {dim}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} antes de la inserción: {cant_antes}")

query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN oricenasicod  TYPE character(1),
ALTER COLUMN cenasicod  TYPE character(3),
ALTER COLUMN arehoscod  TYPE character(2),
ALTER COLUMN servhoscod  TYPE character(3),
ALTER COLUMN estenfcod  TYPE character(2),
ALTER COLUMN habcod  TYPE character(4),
ALTER COLUMN habubides  TYPE character(20),
ALTER COLUMN tiphabcod  TYPE character(1),
ALTER COLUMN esthabcod  TYPE character(1),
ALTER COLUMN habestregcod  TYPE character(1);


INSERT INTO {dim} 

(ori_cas,cas_cod,are_cod,ser_cod,enf_cod,hab_cod,hab_ubi,tip_hab,est_hab,est_reg) 

SELECT 

oricenasicod, cenasicod, arehoscod, servhoscod, estenfcod, habcod, habubides, tiphabcod, esthabcod, habestregcod

FROM tmp_{tabla} t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} d 
    WHERE 
    
    d.ori_cas = t.oricenasicod AND
    d.cas_cod = t.cenasicod AND
    d.are_cod = t.arehoscod AND
    d.ser_cod = t.servhoscod AND
    d.enf_cod = t.estenfcod AND
    d.hab_cod = t.habcod
);
"""



c1= text(query)
cursor=connection4.execute(c1)

#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
SELECT COUNT(*) FROM {dim};
"""

c2= text(query2)
cursor=connection4.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla dim_hoshab antes de la inserción: 16432
Cantidad de filas en la tabla dim_hoshab después de la inserción: 16671
La cantidad de filas insertadas fue de: 239
